1. Análise de Fornecedores

Este projeto tem como objetivo analisar dados de fornecedores, realizando limpeza e extraindo insights para tomada de decisão.

2. Importação de bibliotecas

In [271]:
import pandas as pd
import numpy as np
import unicodedata

3. Leitura dos dados

In [272]:
df_analise = pd.read_csv("analise_fornecedores.csv", encoding='latin1', sep=";", low_memory=False)
display(df_analise)

display(df_analise.info())

,id_pedido,data_pedido,fornecedor,produto,preco_kg,quantidade_kg,qualidade,aceitacao_socios,tempo_entrega_dias,avaliacao_chef,observacao
0,1,05/01/2025,Frigorífico Sul,Picanha,82.0,25,8,9,2.0,9,entrega rápida
1,2,07/01/2025,Carnes Premium,Frango,20.0,40,8,8,3.0,8,NaN
2,3,09/01/2025,Proteína Nobre,Costela,47.0,35,8,7,2.0,8,NaN
3,4,10/01/2025,Frigorifico Sul,Picanha,80.0,20,7,8,2.0,8,erro no nome
4,5,12/01/2025,Distribuidora Boi Forte,Frango,17.0,50,6,7,4.0,6,atraso na entrega
5,6,14/01/2025,Carnes Premium,Picanha,95.0,15,9,9,3.0,9,NaN
6,7,16/01/2025,Proteína Nobre,Frango,19.0,60,7,7,2.0,7,NaN
7,8,18/01/2025,Frigorífico Sul,Costela,45.0,30,8,8,3.0,8,NaN
8,9,20/01/2025,Carnes Premium,Costela,52.0,28,9,9,3.0,9,NaN
9,10,22/01/2025,Proteina Nobre,Picanha,88.0,18,8,8,2.0,8,nome inconsistente


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40 entries, 0 to 39
Data columns (total 11 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id_pedido           40 non-null     int64  
 1   data_pedido         40 non-null     object 
 2   fornecedor          40 non-null     object 
 3   produto             40 non-null     object 
 4   preco_kg            39 non-null     float64
 5   quantidade_kg       40 non-null     int64  
 6   qualidade           40 non-null     int64  
 7   aceitacao_socios    40 non-null     int64  
 8   tempo_entrega_dias  39 non-null     float64
 9   avaliacao_chef      40 non-null     int64  
 10  observacao          8 non-null      object 
dtypes: float64(2), int64(5), object(4)
memory usage: 3.6+ KB


None

#Problemas identificados

- Datas em formato incorreto  
- Valores nulos e inconsistentes  
- Falta de padronização nos fornecedores  

4. Limpeza

In [273]:
#Convertendo datas

df_analise["data_pedido"] = pd.to_datetime(df_analise["data_pedido"], format="%d/%m/%Y")
df_analise = df_analise.sort_values(by="data_pedido", ascending=True)

#Padronização fornecedores

df_analise["fornecedor"] = df_analise["fornecedor"].str.strip().str.title()

import unicodedata

def remover_acentos(texto):
    return unicodedata.normalize("NFKD", texto).encode("ASCII", "ignore").decode("ASCII")

df_analise["fornecedor"] = df_analise["fornecedor"].apply(remover_acentos)

df_analise["fornecedor"].value_counts()

#Tratando valores nuloes

df_analise[(df_analise["fornecedor"] == "Carnes premium") & (df_analise["produto"] == "Frango")]

,id_pedido,data_pedido,fornecedor,produto,preco_kg,quantidade_kg,qualidade,aceitacao_socios,tempo_entrega_dias,avaliacao_chef,observacao


In [274]:
df_analise[(df_analise["fornecedor"] == "Carnes Premium") & (df_analise["produto"] == "Frango")]["preco_kg"].mean()

np.float64(20.333333333333332)

In [275]:
df_analise.loc[
    (df_analise["fornecedor"] == "Carnes premium") &
    (df_analise["produto"] == "Frango") &
    (df_analise["preco_kg"].isna()),
    "preco_kg"
] = 20.33
display(df_analise["preco_kg"].unique())

array([ 82.,  20.,  47.,  80.,  17.,  95.,  19.,  45.,  52.,  88.,  79.,
        nan,  18.,  96., -80.,  51.,  44.,  94.,  78.,  21.,  46.,  53.,
        83., 300.,  89.,  40.])

Tratando valores negativos


In [276]:
df_analise.loc[df_analise["preco_kg"] < 0, "preco_kg"] = None
display(df_analise["preco_kg"].unique())

array([ 82.,  20.,  47.,  80.,  17.,  95.,  19.,  45.,  52.,  88.,  79.,
        nan,  18.,  96.,  51.,  44.,  94.,  78.,  21.,  46.,  53.,  83.,
       300.,  89.,  40.])

In [277]:
display(df_analise[["id_pedido", "fornecedor",	"produto",	"preco_kg",	"quantidade_kg"]])

,id_pedido,fornecedor,produto,preco_kg,quantidade_kg
0,1,Frigorifico Sul,Picanha,82.0,25
1,2,Carnes Premium,Frango,20.0,40
2,3,Proteina Nobre,Costela,47.0,35
3,4,Frigorifico Sul,Picanha,80.0,20
4,5,Distribuidora Boi Forte,Frango,17.0,50
5,6,Carnes Premium,Picanha,95.0,15
6,7,Proteina Nobre,Frango,19.0,60
7,8,Frigorifico Sul,Costela,45.0,30
8,9,Carnes Premium,Costela,52.0,28
9,10,Proteina Nobre,Picanha,88.0,18


In [278]:
df_id_pedido_17= df_analise[df_analise["id_pedido"]==17]
display(df_id_pedido_17)

,id_pedido,data_pedido,fornecedor,produto,preco_kg,quantidade_kg,qualidade,aceitacao_socios,tempo_entrega_dias,avaliacao_chef,observacao
16,17,2025-02-02,Frigorifico Sul,Picanha,NaN,22,8,8,2.0,8,preço negativo


In [279]:
df_analise[(df_analise["fornecedor"] == "Frigorifico Sul") & (df_analise["produto"] == "Picanha")]["preco_kg"].mean()

np.float64(81.66666666666667)

In [280]:
df_analise.loc[
    (df_analise["fornecedor"] == "Frigorifico Sul") &
    (df_analise["produto"] == "Picanha") &
    (df_analise["preco_kg"].isna()),
    "preco_kg"
] = 81.66
display(df_analise["preco_kg"].unique())

array([ 82.  ,  20.  ,  47.  ,  80.  ,  17.  ,  95.  ,  19.  ,  45.  ,
        52.  ,  88.  ,  79.  ,    nan,  18.  ,  96.  ,  81.66,  51.  ,
        44.  ,  94.  ,  78.  ,  21.  ,  46.  ,  53.  ,  83.  , 300.  ,
        89.  ,  40.  ])

In [281]:
preco_medio = df_analise.groupby("fornecedor")["preco_kg"].mean().round(2)
avaliacao = df_analise.groupby("fornecedor")["avaliacao_chef"].mean().round(2)
tempo = df_analise.groupby("fornecedor")["tempo_entrega_dias"].mean().round(2)

resultado = pd.DataFrame({
    "preco_medio": preco_medio,
    "avaliacao": avaliacao,
    "tempo_entrega_dias": tempo
})
resultado

,preco_medio,avaliacao,tempo_entrega_dias
fornecedor,,,
Carnes Premium,81.55,8.67,3.00
Distribuidora Boi Forte,46.71,6.43,4.00
Frigorifico Sul,50.88,7.91,2.36
Proteina Nobre,50.90,7.70,2.00


## Insights

- A **Carnes Premium** apresenta o maior preço médio (R$ 81,55/kg), porém também possui a melhor avaliação (8,67), indicando um posicionamento premium com foco em qualidade.

- A **Distribuidora Boi Forte** possui o menor preço médio (R$ 46,71/kg), mas também a menor avaliação (6,43) e o maior tempo de entrega (4 dias), sugerindo menor qualidade e eficiência.

- O **Frigorífico Sul** apresenta bom equilíbrio entre preço (R$ 50,88/kg), avaliação (7,91) e tempo de entrega (2 dias), sendo uma opção de melhor custo-benefício.

- A **Proteína Nobre** possui o menor tempo médio de entrega (2 dias), com preço e avaliação intermediários, sendo uma boa opção para rapidez.

- De forma geral, observa-se uma relação entre preço e qualidade: fornecedores mais caros tendem a apresentar melhores avaliações.

- Conclusão: a escolha do fornecedor ideal depende do objetivo do negócio: redução de custos, qualidade do produto ou rapidez na entrega.